🥇 top_crypto_assets

In [0]:
# Load Silver Tables

df_markets = spark.read.table("medallionarchitecture.silver.coin_markets")
df_price   = spark.read.table("medallionarchitecture.silver.simple_price")
df_details = spark.read.table("medallionarchitecture.silver.coin_details")

In [0]:
# join

from pyspark.sql.functions import col

df_joined = df_markets.alias("m") \
    .join(df_price.alias("p"), col("m.coin_id") == col("p.coin"), "left") \
    .join(df_details.alias("d"), col("m.coin_id") == col("d.coin_id"), "left")

In [0]:
# the selection of business fields

df_gold = df_joined.select(
    col("m.coin_id"),
    col("m.name"),
    col("m.symbol"),

    col("m.current_price"),
    col("p.price_usd").alias("api_price_usd"),

    col("m.market_cap"),
    col("m.total_volume"),

    col("d.category"),

    col("m.ingestion_time")
)

In [0]:
# DQ Check

df_gold = df_gold.filter(col("market_cap").isNotNull())
df_gold = df_gold.dropDuplicates(["coin_id", "category"])

In [0]:
# save gold 

df_gold.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallionarchitecture.gold.top_crypto_assets")